# Multi-Agent Patterns: Supervisor, Pipeline, and Debate

When tasks exceed what a single agent can reliably handle, multi-agent architectures distribute the work across specialized agents. Each pattern trades different costs: coordination overhead, trust complexity, and latency.

## Why Multiple Agents

Single agents face limits:
- Context window: a long task generates more intermediate state than fits in one context
- Specialization: a generalist agent performing a complex task is less reliable than multiple specialists
- Parallelism: independent subtasks can run concurrently
- Verification: an agent reviewing its own work is biased; a separate agent provides independent review

The key cost: multi-agent coordination introduces new failure modes (miscommunication between agents, trust violations, cascading failures) that single-agent systems don't have.

In [ ]:
from dataclasses import dataclass, field
from typing import Callable
import json

@dataclass
class AgentMessage:
    sender: str
    recipient: str
    content: str
    message_type: str = 'task'  # 'task', 'result', 'error', 'query'

@dataclass
class AgentResult:
    agent: str
    output: str
    success: bool
    tokens_used: int = 0

class SpecializedAgent:
    def __init__(self, name: str, model_fn: Callable, system_prompt: str):
        self.name = name
        self.model = model_fn
        self.system = system_prompt

    def process(self, task: str) -> AgentResult:
        prompt = f'{self.system}\n\nTask: {task}'
        output = self.model(prompt)
        return AgentResult(agent=self.name, output=output,
                            success='error' not in output.lower(), tokens_used=len(output.split()))

class SupervisorAgent:
    def __init__(self, supervisor_fn: Callable, workers: dict):
        self.supervisor = supervisor_fn
        self.workers = workers
        self.message_log: list = []

    def _parse_delegation(self, text: str):
        import re
        m = re.search(r'DELEGATE:(\w+):(.+?)(?=DELEGATE:|FINAL:|$)', text, re.DOTALL)
        f = re.search(r'FINAL:(.+)', text, re.DOTALL)
        if m:
            return m.group(1), m.group(2).strip(), None
        if f:
            return None, None, f.group(1).strip()
        return None, None, None

    def run(self, task: str) -> dict:
        context = f'Workers: {list(self.workers.keys())}\nTask: {task}'
        results = {}
        for _ in range(5):
            response = self.supervisor(context)
            worker, subtask, final = self._parse_delegation(response)
            if final:
                return {'final': final, 'results': results, 'steps': len(results)}
            if worker and worker in self.workers:
                result = self.workers[worker].process(subtask)
                results[worker] = result
                self.message_log.append(AgentMessage(worker, 'supervisor', result.output, 'result'))
                context += f'\n[{worker} result]: {result.output[:100]}'
        return {'final': 'incomplete', 'results': results, 'steps': len(results)}

# Demo: research + write pipeline
def mock_researcher(prompt): return 'Research found: Python is the top ML language with 70% adoption.'
def mock_writer(prompt): return 'Report: Python dominates ML with 70% adoption according to recent surveys.'
def mock_supervisor(context):
    if 'result' not in context:
        return 'DELEGATE:researcher:Find current ML programming language statistics'
    if 'writer' not in context.lower():
        return 'DELEGATE:writer:Write a brief report on ML language adoption'
    return 'FINAL: Report complete — Python leads ML at 70% adoption.'

workers = {
    'researcher': SpecializedAgent('researcher', mock_researcher, 'You are a research specialist.'),
    'writer': SpecializedAgent('writer', mock_writer, 'You are a technical writer.'),
}
supervisor = SupervisorAgent(mock_supervisor, workers)
result = supervisor.run('Create a brief report on ML programming language trends')
print(f'Final: {result["final"]}')
print(f'Steps: {result["steps"]}')
for k, v in result['results'].items():
    print(f'  {k}: {v.output[:80]}')

## Pipeline Pattern

The pipeline pattern passes work through a fixed sequence of agents: output of agent N is input to agent N+1. Useful for:
- Multi-stage document processing (extract → summarize → format)
- Code generation pipelines (spec → code → test → review)
- Data transformation chains

Advantage: predictable, easy to debug, easy to add/remove stages. Disadvantage: rigid — cannot adapt the flow based on intermediate results.

In [ ]:
class AgentPipeline:
    def __init__(self, stages: list):
        self.stages = stages  # list of (name, agent_fn) tuples

    def run(self, initial_input: str) -> dict:
        current = initial_input
        stage_outputs = []
        for stage_name, agent_fn in self.stages:
            output = agent_fn(current)
            stage_outputs.append({'stage': stage_name, 'output': output[:100]})
            current = output
        return {'final_output': current, 'stages': stage_outputs}

# Code generation pipeline
stages = [
    ('spec_writer', lambda t: f'Specification: Write a function that {t}. Return type: str. Handle edge cases.'),
    ('code_gen', lambda s: f'def solution(n):\n    # {s[:50]}\n    return str(n)  # placeholder implementation'),
    ('test_writer', lambda c: f'def test_solution():\n    assert solution(5) == "5"\n    assert solution(0) == "0"'),
    ('reviewer', lambda t: f'Code review: Implementation looks correct. Tests cover basic cases. Consider edge cases for None input.'),
]
pipeline = AgentPipeline(stages)
result = pipeline.run('converts an integer to its string representation')
print('Pipeline execution:')
for s in result['stages']:
    print(f'  [{s["stage"]}]: {s["output"]}')

## Debate and Mixture-of-Agents

**Debate** (Irving et al. 2018 / Du et al. 2023): two agents argue opposing positions; a judge selects the winner or synthesizes. Reduces single-model bias and improves factual accuracy on controversial or ambiguous questions.

**Mixture-of-Agents** (Wang et al. 2024): multiple agents independently answer; a synthesizer combines the best elements from each. Outperforms any individual agent, especially at smaller model sizes. The diversity of independent approaches reduces correlated errors.

Both patterns require a reliable judge or synthesizer — if the arbitrator is biased or weak, the multi-agent overhead produces no benefit.